# Sprint 12 - APIs REST (Nivell 1)

Tasca S12.01

En este nivel pruebo la libreria `requests` haciendo peticiones a JSONPlaceholder, que es una API
publica de pruebas. Voy mirando el codigo de estado de cada peticion y lo que me devuelve.

### Enunciat (Nivell 1)

Des d'un Jupyter Notebook faras els seguents exercicis utilitzant la llibreria requests de Python.

1. Consulta l'API publica JSONPlaceholder amb el metode GET per obtenir la llista de publicacions (/posts), d'usuaris (/users) i de tasques (/todos).
2. Mostra per pantalla la quantitat total de cada recurs i el codi d'estat de cada peticio.
3. Fes una peticio a una publicacio inexistent per obtenir un error 404 i mostra el codi d'estat.
4. Fes una peticio POST per crear una nova publicacio ficticia (titol, cos i userId). Mostra la resposta JSON i el codi d'estat.
5. Fes una peticio PATCH per modificar parcialment una publicacio. Mostra la resposta i el codi d'estat.
6. Fes una peticio DELETE sobre una publicacio. Mostra la resposta i el codi d'estat.

In [2]:
import requests
import json

### Ejercicios 1 y 2 - GET de los tres recursos

Hago un GET a cada recurso y miro cuantos elementos tiene y el codigo de estado.

In [3]:
posts = requests.get("https://jsonplaceholder.typicode.com/posts")
users = requests.get("https://jsonplaceholder.typicode.com/users")
todos = requests.get("https://jsonplaceholder.typicode.com/todos")

print("posts:", len(posts.json()), "elementos - estado", posts.status_code)
print("users:", len(users.json()), "elementos - estado", users.status_code)
print("todos:", len(todos.json()), "elementos - estado", todos.status_code)

posts: 100 elementos - estado 200
users: 10 elementos - estado 200
todos: 200 elementos - estado 200


### Ejercicio 3 - una publicacion que no existe (404)

Si pido un post que no existe me tiene que dar un 404.

In [4]:
r = requests.get("https://jsonplaceholder.typicode.com/posts/9999")
print("codigo de estado:", r.status_code)

codigo de estado: 404


### Ejercicio 4 - crear un post con POST

Mando un post nuevo. Me devuelve el objeto creado con un id nuevo (el 101) y el estado 201.

In [5]:
nuevo = {
    "title": "prueba de post",
    "body": "esto es solo una prueba para el sprint",
    "userId": 1
}

r = requests.post("https://jsonplaceholder.typicode.com/posts", json=nuevo)
print("estado:", r.status_code)
print(r.json())

estado: 201
{'title': 'prueba de post', 'body': 'esto es solo una prueba para el sprint', 'userId': 1, 'id': 101}


### Ejercicio 5 - modificar con PATCH

Con PATCH cambio solo el titulo del post 1.

In [6]:
r = requests.patch("https://jsonplaceholder.typicode.com/posts/1", json={"title": "titulo cambiado"})
print("estado:", r.status_code)
print(r.json())

estado: 200
{'userId': 1, 'id': 1, 'title': 'titulo cambiado', 'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'}


### Ejercicio 6 - borrar con DELETE

Borro el post 1. Devuelve un objeto vacio y estado 200.

In [7]:
r = requests.delete("https://jsonplaceholder.typicode.com/posts/1")
print("estado:", r.status_code)
print(r.json())

estado: 200
{}


En resumen he probado GET, POST, PATCH y DELETE y he visto los codigos de estado:
200 cuando va bien, 404 cuando no existe y 201 al crear con POST.

# Sprint 12 - APIs REST (Nivell 2)

Tasca S12.01

Para este nivel he elegido la API de Open-Meteo, que da informacion del tiempo y es gratis y
sin clave. Hago una peticion GET, miro la respuesta JSON y al final lo paso todo a un DataFrame.

### Enunciat (Nivell 2)

Interaccio amb una API publica real.

1. Explora el repositori de Public APIs i tria una API que permeti fer peticions GET.
2. Llegeix la documentacio de l'API: apunta en un markdown com a minim dos endpoints diferents, mira si te filtres o parametres opcionals i comprova que la resposta sigui en JSON.
3. Fes una peticio GET senzilla: mostra el codi d'estat i imprimeix alguns camps de la resposta JSON.
4. Converteix les dades a un DataFrame de pandas i mostra les primeres files.

### Documentacion de la API (Open-Meteo)

Web: https://open-meteo.com/

**Dos endpoints que he mirado:**

- `/v1/forecast` -> prevision del tiempo (actual y por horas) a partir de unas coordenadas.
- `/v1/air-quality` (en el servidor air-quality-api.open-meteo.com) -> calidad del aire (PM2.5, PM10...).

**Algunos parametros del endpoint forecast:**

- `latitude` y `longitude`: las coordenadas del sitio.
- `current`: las variables del momento actual (temperatura, viento...).
- `hourly`: las variables hora a hora (temperatura, lluvia, humedad...).
- `timezone`: la zona horaria.
- `forecast_days`: cuantos dias quiero (de 1 a 16).

La respuesta viene en JSON, que es lo que nos pide el ejercicio.

### Ejercicio 3 - peticion GET

Pido el tiempo de Barcelona. Le paso las coordenadas y las variables que quiero en los parametros.

In [8]:
import requests
import pandas as pd

url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": 41.3874,
    "longitude": 2.1686,
    "current": "temperature_2m,relative_humidity_2m,apparent_temperature,wind_speed_10m,weather_code",
    "hourly": "temperature_2m,relative_humidity_2m,precipitation",
    "timezone": "Europe/Madrid",
    "forecast_days": 3
}

r = requests.get(url, params=params)
print("codigo de estado:", r.status_code)

datos = r.json()

codigo de estado: 200


Imprimo algunos campos de la respuesta para ver que ha llegado bien.

In [9]:
print("latitud:", datos["latitude"])
print("longitud:", datos["longitude"])
print("zona horaria:", datos["timezone"])

actual = datos["current"]
print()
print("ahora mismo en Barcelona:")
print("hora:", actual["time"])
print("temperatura:", actual["temperature_2m"], "C")
print("sensacion termica:", actual["apparent_temperature"], "C")
print("humedad:", actual["relative_humidity_2m"], "%")
print("viento:", actual["wind_speed_10m"], "km/h")

latitud: 41.375
longitud: 2.125
zona horaria: Europe/Madrid

ahora mismo en Barcelona:
hora: 2026-06-09T11:30
temperatura: 24.0 C
sensacion termica: 25.6 C
humedad: 66 %
viento: 6.8 km/h


### Ejercicio 4 - pasar los datos a un DataFrame

Los datos por horas vienen en listas separadas (una para las horas, otra para la temperatura, etc).
Como tienen la misma longitud puedo meter el diccionario `hourly` directamente en un DataFrame.

In [10]:
por_horas = datos["hourly"]
df = pd.DataFrame(por_horas)

# paso la columna de tiempo a tipo fecha
df["time"] = pd.to_datetime(df["time"])

print(df.shape)
df.head()

(72, 4)


,time,temperature_2m,relative_humidity_2m,precipitation
0,2026-06-09 00:00:00,21.3,86,0.0
1,2026-06-09 01:00:00,21.0,88,0.0
2,2026-06-09 02:00:00,20.9,92,0.0
3,2026-06-09 03:00:00,20.7,91,0.0
4,2026-06-09 04:00:00,20.6,90,0.0


In [11]:
# unas estadisticas rapidas de la temperatura
df["temperature_2m"].describe()

count    72.000000
mean     21.347222
std       1.620987
min      18.000000
25%      20.200000
50%      21.250000
75%      22.325000
max      24.700000
Name: temperature_2m, dtype: float64

Con esto ya he hecho la peticion GET a una API real, he visto el codigo de estado y algunos
campos del JSON, y he pasado los datos por horas a un DataFrame de pandas.